# Supplementary studies


Four supplementary studies use fixed designs. Reuse of the core TEST partitions makes the forecasting results exploratory; PPGR is a separate meal-level task and an evidence-bound partial reconstruction.

| Study | Inputs and comparison | Main constraint |
|---|---|---|
| Microbiome | Binary subject-level detection indicators; residual correction against the same-cohort full Ridge reference. | TRAIN-only prevalence screening; missing is not absence; windows are not independent microbiome observations. |
| Device/horizon | Dexcom at 30 min and Libre at 60 min; GlucoBench LR and persistence. | Rebuild task-specific windows and fit preprocessing on that task's TRAIN only. |
| PPGR | Breakfast AUC/iAUC, 19 predictors, XGBoost 1.6.2 and LOSO. | Published and author-code rules are distinguished below; correlations are descriptive references. |
| GRU/TCN | The same 25-value Libre histories and admitted windows, five seeds and validation early stopping. | Fixed architectures and budgets; report every seed and compare on matched windows. |


In [1]:
# --- Import optional-study dependencies ---

import json
import os
import platform
import subprocess
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from scipy.stats import pearsonr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.baselines import fit_glucobench, gather_lags, predict_glucobench
from src.evaluation import draw_subjects
from src.paths import DATA_ROOT, PROJECT_ROOT, validate_data_root
from src.windows import assign_splits, build_windows

# --- Configure artifact directories and evaluation protocols ---


validate_data_root()
DATA_QUALITY_DIR = PROJECT_ROOT / "results" / "data_quality"
ALIGNED_DIR = PROJECT_ROOT / "results" / "aligned_data"
WINDOWS_DIR = PROJECT_ROOT / "results" / "forecast_windows"
BASELINE_DIR = PROJECT_ROOT / "results" / "baselines"
MANUAL_DIR = PROJECT_ROOT / "results" / "manual_ablation"
ART06 = PROJECT_ROOT / "results" / "manual_evaluation"
EXT_DIR = PROJECT_ROOT / "results" / "supplementary_studies"
EXT_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL_A = "Within-subject Future Prediction"
PROTOCOL_B = "Unseen-subject Generalization"
PROTOCOLS = [PROTOCOL_A, PROTOCOL_B]

# --- Configure compact tabular displays ---

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 25)


# --- Subject-level metrics shared by the studies ---

def subject_metric_summary(frame, prediction_column):
    evaluated = frame[["subject", "target_value", prediction_column]].copy()
    evaluated["absolute_error"] = (evaluated["target_value"] - evaluated[prediction_column]).abs()
    evaluated["squared_error"] = (evaluated["target_value"] - evaluated[prediction_column]) ** 2
    by_subject = (
        evaluated.groupby("subject", observed=True)
        .agg(mae=("absolute_error", "mean"), mse=("squared_error", "mean"), windows=("subject", "size"))
        .reset_index()
    )
    by_subject["rmse"] = np.sqrt(by_subject["mse"])
    summary = {
        "subjects": int(by_subject["subject"].nunique()),
        "windows": int(len(evaluated)),
        "subject_mean_mae": float(by_subject["mae"].mean()),
        "subject_mean_rmse": float(by_subject["rmse"].mean()),
        "pooled_mae": float(evaluated["absolute_error"].mean()),
        "pooled_rmse": float(np.sqrt(evaluated["squared_error"].mean())),
    }
    return summary, by_subject[["subject", "windows", "mae", "rmse"]]

## Microbiome residual analysis

Test whether subject-level binary detection indicators improve the same-cohort full Ridge reference. Fit residual correction at subject level so repeated glucose windows do not become independent microbiome observations.


In [2]:
# --- Define inputs and microbiome-screening parameters ---

microbiome_paths = {
    "task_protocol": WINDOWS_DIR / "task_protocol.json",
    "fold_assignments": WINDOWS_DIR / "fold_assignments.parquet",
    "glucose_history_samples": WINDOWS_DIR / "glucose_history_samples.parquet",
    "glucose_feature_registry": WINDOWS_DIR / "feature_registry.csv",
    "microbiome_canonical": ALIGNED_DIR / "microbiome_canonical.parquet",
    "static_canonical": ALIGNED_DIR / "static_canonical.parquet",
    "comparison_cohorts": MANUAL_DIR / "comparison_cohorts.parquet",
    "activity_diet_features": MANUAL_DIR / "activity_diet_features.parquet",
    "activity_diet_registry": MANUAL_DIR / "activity_diet_feature_registry.csv",
    "static_features": MANUAL_DIR / "static_features.parquet",
    "static_feature_coverage": MANUAL_DIR / "static_feature_coverage.csv",
    "ablation_fits": MANUAL_DIR / "ablation_fits.csv",
}

# --- Validate the primary-task definition ---

task_protocol = json.loads(microbiome_paths["task_protocol"].read_text(encoding="utf-8"))
primary_task = task_protocol["primary_task"]
assert primary_task["task_id"] == "primary_libre_30min"
assert primary_task["target_field"] == "libre_gl"
assert primary_task["forecast_horizon_minutes"] == 30

# --- Fix microbiome-screening settings before validation and test evaluation ---

METHOD_VERSION = "microbiome_subject_residual_v1"

MIN_DETECTED_TRAIN_SUBJECTS = 3
MIN_NOT_DETECTED_TRAIN_SUBJECTS = 3
MIN_OBSERVED_FRACTION = 0.80
MICROBIOME_ALPHA_GRID = (10.0, 100.0, 1000.0)
KEY = ["task_id", "subject", "origin_time"]

In [3]:
# --- Build the microbiome-and-Gut common cohort with baseline features ---

microbiome = pd.read_parquet(microbiome_paths["microbiome_canonical"])
comparison = pd.read_parquet(microbiome_paths["comparison_cohorts"])
fold_assignments = pd.read_parquet(microbiome_paths["fold_assignments"])

metadata_columns = {
    "subject", "static_source_table", "static_source_relative_path", "in_dynamic_cohort",
    "static_record_present", "nonmissing_value_count", "all_value_fields_missing",
    "static_availability_status",
}
microbiome_fields = [column for column in microbiome.columns if column not in metadata_columns]
assert len(microbiome_fields) == 1979

# Availability flags come from the data alignment static tables.
static_availability = pd.read_parquet(microbiome_paths["static_canonical"], columns=["subject", "microbiome_values_available", "gut_values_available"])
comparison = comparison.merge(static_availability, on="subject", how="left", validate="many_to_one")
comparison[["microbiome_values_available", "gut_values_available"]] = comparison[
    ["microbiome_values_available", "gut_values_available"]
].fillna(False)
comparison["cohort_microbiome_gut"] = (
    comparison["cohort_core"].astype(bool)
    & comparison["microbiome_values_available"].astype(bool)
    & comparison["gut_values_available"].astype(bool)
)

coverage = (
    comparison.groupby(["evaluation_protocol", "split"], observed=True)
    .agg(
        available_windows=("subject", "size"),
        available_subjects=("subject", "nunique"),
        core_windows=("cohort_core", "sum"),
        microbiome_gut_windows=("cohort_microbiome_gut", "sum"),
    )
    .reset_index()
)
joint_subject_counts = (
    comparison.loc[comparison["cohort_microbiome_gut"]]
    .groupby(["evaluation_protocol", "split"], observed=True)["subject"]
    .nunique()
    .rename("microbiome_gut_subjects")
    .reset_index()
)
coverage = coverage.merge(
    joint_subject_counts, on=["evaluation_protocol", "split"], how="left", validate="one_to_one"
)
coverage["microbiome_gut_subjects"] = coverage["microbiome_gut_subjects"].fillna(0).astype(int)
coverage["coverage_fraction_of_core_windows"] = (
    coverage["microbiome_gut_windows"] / coverage["core_windows"]
)

glucose_registry = pd.read_csv(microbiome_paths["glucose_feature_registry"])
activity_diet_registry = pd.read_csv(microbiome_paths["activity_diet_registry"])
static_coverage = pd.read_csv(microbiome_paths["static_feature_coverage"])
g_features = glucose_registry["feature_name"].tolist()
a_features = activity_diet_registry.loc[
    activity_diet_registry["group_code"].eq("A") & activity_diet_registry["variant"].eq("primary"),
    "feature_name",
].tolist()
d_features = activity_diet_registry.loc[
    activity_diet_registry["group_code"].eq("D") & activity_diet_registry["variant"].eq("primary"),
    "feature_name",
].tolist()
b_features = static_coverage.loc[
    static_coverage["record_type"].eq("feature")
    & static_coverage["group_code"].eq("B")
    & static_coverage["role"].eq("core")
    & ~static_coverage["feature_name"].eq("bio_values_available"),
    "feature_name",
].tolist()
base_features = g_features + a_features + d_features + b_features
assert len(base_features) == 60 and len(set(base_features)) == len(base_features)

ablation_fits = pd.read_csv(microbiome_paths["ablation_fits"])
full_model_selected = ablation_fits.loc[
    ablation_fits["scope"].eq("summary")
    & ablation_fits["config_id"].eq("G+A+D+B")
    & ablation_fits["cohort"].eq("cohort_core")
    & ablation_fits["family"].eq("ridge")
    & ablation_fits["selected"].eq(True),
    ["evaluation_protocol", "alpha"],
].drop_duplicates()
fixed_alpha_by_protocol = {row.evaluation_protocol: float(row.alpha) for row in full_model_selected.itertuples(index=False)}
assert set(fixed_alpha_by_protocol) == set(PROTOCOLS)
samples = pd.read_parquet(microbiome_paths["glucose_history_samples"], columns=KEY + ["target_time", "target_value"] + g_features)
activity_diet = pd.read_parquet(microbiome_paths["activity_diet_features"], columns=KEY + a_features + d_features)
static_features = pd.read_parquet(microbiome_paths["static_features"], columns=["subject"] + b_features)

matched = comparison.loc[comparison["cohort_microbiome_gut"], ["evaluation_protocol", "split"] + KEY].copy()
model_data = (
    matched.merge(samples, on=KEY, how="left", validate="many_to_one")
    .merge(activity_diet, on=KEY, how="left", validate="many_to_one")
    .merge(static_features, on="subject", how="left", validate="many_to_one")
)
assert model_data.groupby(["evaluation_protocol"] + KEY, observed=True).size().eq(1).all()
assert model_data["target_value"].notna().all()


In [4]:
# --- Screen microbiome indicators using training subjects only ---

def screen_microbiome(training_subjects):
    """Apply a fixed prevalence/observedness rule using one row per training subject."""
    train_subjects = sorted(set(training_subjects))
    table = microbiome.loc[microbiome["subject"].isin(train_subjects), ["subject"] + microbiome_fields].copy()
    assert table["subject"].is_unique and set(table["subject"]) == set(train_subjects)
    n_train = len(train_subjects)
    observed = table[microbiome_fields].notna().sum(axis=0)
    detected = table[microbiome_fields].eq(1.0).sum(axis=0)
    not_detected = table[microbiome_fields].eq(0.0).sum(axis=0)
    min_observed = int(np.ceil(MIN_OBSERVED_FRACTION * n_train))
    selected = (
        observed.ge(min_observed)
        & detected.ge(MIN_DETECTED_TRAIN_SUBJECTS)
        & not_detected.ge(MIN_NOT_DETECTED_TRAIN_SUBJECTS)
    )
    reason = np.select(
        [
            observed.lt(min_observed),
            detected.eq(0),
            not_detected.eq(0),
            detected.lt(MIN_DETECTED_TRAIN_SUBJECTS),
            not_detected.lt(MIN_NOT_DETECTED_TRAIN_SUBJECTS),
        ],
        ["insufficient_observed_subjects", "constant_not_detected", "constant_detected", "low_detection", "near_universal"],
        default="retained",
    )
    detail = pd.DataFrame({
        "feature_name": microbiome_fields,
        "training_subjects": n_train,
        "observed_training_subjects": observed.reindex(microbiome_fields).to_numpy(),
        "detected_training_subjects": detected.reindex(microbiome_fields).to_numpy(),
        "not_detected_training_subjects": not_detected.reindex(microbiome_fields).to_numpy(),
        "training_prevalence": (detected / observed.replace(0, np.nan)).reindex(microbiome_fields).to_numpy(),
        "selected": selected.reindex(microbiome_fields).to_numpy(),
        "exclusion_reason": reason,
    })
    return detail


# --- Assess screening stability across training-subject omissions ---

screening_tables = []
stability_rows = []
selected_features_by_protocol = {}
for protocol in PROTOCOLS:
    training_subjects = sorted(
        model_data.loc[
            model_data["evaluation_protocol"].eq(protocol) & model_data["split"].eq("train"), "subject"
        ].unique()
    )
    full_screen = screen_microbiome(training_subjects)
    selected_set = set(full_screen.loc[full_screen["selected"], "feature_name"])
    assert selected_set
    selected_features_by_protocol[protocol] = sorted(selected_set)

    loo_sets = []
    for omitted_subject in training_subjects:
        loo_screen = screen_microbiome([subject for subject in training_subjects if subject != omitted_subject])
        loo_sets.append(set(loo_screen.loc[loo_screen["selected"], "feature_name"]))
    loo_jaccard = [
        len(selected_set & subset) / len(selected_set | subset) if selected_set | subset else 1.0
        for subset in loo_sets
    ]
    loo_frequency = {
        feature: np.mean([feature in subset for subset in loo_sets]) for feature in microbiome_fields
    }
    full_screen["evaluation_protocol"] = protocol
    full_screen["leave_one_subject_out_selection_frequency"] = full_screen["feature_name"].map(loo_frequency)
    screening_tables.append(full_screen)
    stability_rows.append({
        "evaluation_protocol": protocol,
        "training_subjects": len(training_subjects),
        "retained_features": len(selected_set),
        "loo_retained_features_min": min(map(len, loo_sets)),
        "loo_retained_features_median": float(np.median(list(map(len, loo_sets)))),
        "loo_retained_features_max": max(map(len, loo_sets)),
        "full_vs_loo_jaccard_mean": float(np.mean(loo_jaccard)),
        "full_vs_loo_jaccard_min": float(np.min(loo_jaccard)),
        "full_vs_loo_jaccard_max": float(np.max(loo_jaccard)),
        "always_retained_features": int(sum(value == 1.0 for value in loo_frequency.values())),
    })

microbiome_feature_screening = pd.concat(screening_tables, ignore_index=True)
screen_stability = pd.DataFrame(stability_rows)
set_a, set_b = (set(selected_features_by_protocol[protocol]) for protocol in PROTOCOLS)
cross_protocol_jaccard = len(set_a & set_b) / len(set_a | set_b)

assert not any(
    subject in set(model_data.loc[
        model_data["evaluation_protocol"].eq("Unseen-subject Generalization")
        & model_data["split"].eq("test"), "subject"
    ])
    for subject in model_data.loc[
        model_data["evaluation_protocol"].eq("Unseen-subject Generalization")
        & model_data["split"].eq("train"), "subject"
    ].unique()
)


In [5]:
# --- Define residual-model pipelines ---

def make_base_pipeline(alpha):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha)),
    ])


def make_microbiome_pipeline(alpha):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha)),
    ])


# --- Fit validation-selected microbiome residual models ---

development_data = model_data.loc[~model_data["split"].eq("test")].copy()
fitted_states = {}
alpha_selection_rows = []
for protocol in PROTOCOLS:
    protocol_data = development_data.loc[development_data["evaluation_protocol"].eq(protocol)].copy()
    train = protocol_data.loc[protocol_data["split"].eq("train")].copy()
    validation = protocol_data.loc[protocol_data["split"].eq("validation")].copy()
    assert len(train) and len(validation)

    base_model = make_base_pipeline(fixed_alpha_by_protocol[protocol])
    base_model.fit(train[base_features], train["target_value"])
    train["baseline_prediction"] = base_model.predict(train[base_features])
    validation["baseline_prediction"] = base_model.predict(validation[base_features])

    residual_by_subject = (
        train.assign(residual=train["target_value"] - train["baseline_prediction"])
        .groupby("subject", observed=True)["residual"].mean()
        .rename("mean_training_residual")
        .reset_index()
    )
    selected_features = selected_features_by_protocol[protocol]
    subject_micro = microbiome[["subject"] + selected_features].copy()
    correction_train = residual_by_subject.merge(subject_micro, on="subject", how="left", validate="one_to_one")
    validation_subjects = pd.DataFrame({"subject": sorted(validation["subject"].unique())})
    correction_validation = validation_subjects.merge(subject_micro, on="subject", how="left", validate="one_to_one")
    assert correction_train[selected_features].notna().any(axis=1).all()
    assert correction_validation[selected_features].notna().any(axis=1).all()

    candidates = []
    for alpha in MICROBIOME_ALPHA_GRID:
        micro_model = make_microbiome_pipeline(alpha)
        micro_model.fit(correction_train[selected_features], correction_train["mean_training_residual"])
        correction_map = dict(zip(
            correction_validation["subject"],
            micro_model.predict(correction_validation[selected_features]),
        ))
        candidate = validation.copy()
        candidate["microbiome_correction"] = candidate["subject"].map(correction_map)
        candidate["extension_prediction"] = candidate["baseline_prediction"] + candidate["microbiome_correction"]
        summary, _ = subject_metric_summary(candidate, "extension_prediction")
        candidates.append((summary["subject_mean_rmse"], alpha, micro_model, summary))
        alpha_selection_rows.append({
            "evaluation_protocol": protocol,
            "alpha": alpha,
            "validation_subject_mean_mae": summary["subject_mean_mae"],
            "validation_subject_mean_rmse": summary["subject_mean_rmse"],
            "validation_subjects": summary["subjects"],
            "validation_windows": summary["windows"],
        })

    candidates.sort(key=lambda item: (item[0], item[1]))
    selected_rmse, selected_alpha, selected_micro_model, _ = candidates[0]
    fitted_states[protocol] = {
        "base_model": base_model,
        "micro_model": selected_micro_model,
        "selected_alpha": selected_alpha,
        "selected_features": selected_features,
    }

alpha_selection = pd.DataFrame(alpha_selection_rows)

In [6]:
# Evaluate microbiome residual models across train, validation, and test splits.
prediction_frames = []
performance_rows = []
gain_rows = []
for protocol in PROTOCOLS:
    state = fitted_states[protocol]
    protocol_data = model_data.loc[model_data["evaluation_protocol"].eq(protocol)].copy()
    subject_micro = microbiome[["subject"] + state["selected_features"]].copy()
    unique_subjects = pd.DataFrame({"subject": sorted(protocol_data["subject"].unique())})
    subject_micro = unique_subjects.merge(subject_micro, on="subject", how="left", validate="one_to_one")
    correction_map = dict(zip(
        subject_micro["subject"],
        state["micro_model"].predict(subject_micro[state["selected_features"]]),
    ))

    protocol_data["baseline_prediction"] = state["base_model"].predict(protocol_data[base_features])
    protocol_data["microbiome_correction"] = protocol_data["subject"].map(correction_map)
    protocol_data["extension_prediction"] = (
        protocol_data["baseline_prediction"] + protocol_data["microbiome_correction"]
    )
    assert protocol_data[["baseline_prediction", "microbiome_correction", "extension_prediction"]].notna().all().all()

    for split in ["train", "validation", "test"]:
        split_data = protocol_data.loc[protocol_data["split"].eq(split)].copy()
        baseline_summary, baseline_subjects = subject_metric_summary(split_data, "baseline_prediction")
        extension_summary, extension_subjects = subject_metric_summary(split_data, "extension_prediction")
        for model_name, summary in [("Ridge baseline (G+A+D+B)", baseline_summary), ("Ridge baseline + Microbiome correction", extension_summary)]:
            performance_rows.append({
                "evaluation_protocol": protocol,
                "split": split,
                "model": model_name,
                **summary,
            })
        paired = baseline_subjects.merge(
            extension_subjects, on=["subject", "windows"], suffixes=("_baseline", "_extension"), validate="one_to_one"
        )
        gain_rows.append({
            "evaluation_protocol": protocol,
            "split": split,
            "subjects": int(len(paired)),
            "windows": int(len(split_data)),
            "delta_subject_mean_mae": float((paired["mae_extension"] - paired["mae_baseline"]).mean()),
            "delta_subject_mean_rmse": float((paired["rmse_extension"] - paired["rmse_baseline"]).mean()),
            "subjects_improved_mae": int((paired["mae_extension"] < paired["mae_baseline"]).sum()),
            "subjects_improved_rmse": int((paired["rmse_extension"] < paired["rmse_baseline"]).sum()),
        })

    prediction_frames.append(protocol_data[
        ["evaluation_protocol", "split"] + KEY + ["target_time", "target_value", "baseline_prediction", "microbiome_correction", "extension_prediction"]
    ])

microbiome_extension_predictions = pd.concat(prediction_frames, ignore_index=True)
performance_summary = pd.DataFrame(performance_rows)
gain_summary = pd.DataFrame(gain_rows)

assert microbiome_extension_predictions.groupby(["evaluation_protocol"] + KEY, observed=True).size().eq(1).all()

In [7]:
# Assemble and persist microbiome extension artifacts.
result_rows = []
for row in coverage.itertuples(index=False):
    result_rows.append({
        "record_type": "coverage",
        "evaluation_protocol": row.evaluation_protocol,
        "split": row.split,
        "subjects": row.microbiome_gut_subjects,
        "windows": row.microbiome_gut_windows,
        "core_windows": row.core_windows,
        "coverage_fraction": row.coverage_fraction_of_core_windows,
        "status": "joint Gut+Microbiome common cohort recomputed",
    })

for row in screen_stability.itertuples(index=False):
    protocol_screen = microbiome_feature_screening.loc[
        microbiome_feature_screening["evaluation_protocol"].eq(row.evaluation_protocol)
    ]
    result_rows.append({
        "record_type": "feature_screen",
        "evaluation_protocol": row.evaluation_protocol,
        "split": "train",
        "subjects": row.training_subjects,
        "total_microbiome_fields": len(microbiome_fields),
        "retained_features": row.retained_features,
        "constant_not_detected_excluded": int(protocol_screen["exclusion_reason"].eq("constant_not_detected").sum()),
        "constant_detected_excluded": int(protocol_screen["exclusion_reason"].eq("constant_detected").sum()),
        "low_detection_excluded": int(protocol_screen["exclusion_reason"].eq("low_detection").sum()),
        "near_universal_excluded": int(protocol_screen["exclusion_reason"].eq("near_universal").sum()),
        "insufficient_observed_excluded": int(protocol_screen["exclusion_reason"].eq("insufficient_observed_subjects").sum()),
        "loo_retained_features_min": row.loo_retained_features_min,
        "loo_retained_features_median": row.loo_retained_features_median,
        "loo_retained_features_max": row.loo_retained_features_max,
        "full_vs_loo_jaccard_mean": row.full_vs_loo_jaccard_mean,
        "full_vs_loo_jaccard_min": row.full_vs_loo_jaccard_min,
        "full_vs_loo_jaccard_max": row.full_vs_loo_jaccard_max,
        "always_retained_features": row.always_retained_features,
        "cross_protocol_jaccard": cross_protocol_jaccard,
        "status": "training-subject-only screen",
    })

for row in alpha_selection.itertuples(index=False):
    result_rows.append({
        "record_type": "alpha_selection",
        "evaluation_protocol": row.evaluation_protocol,
        "split": "validation",
        "alpha": row.alpha,
        "subjects": row.validation_subjects,
        "windows": row.validation_windows,
        "subject_mean_mae": row.validation_subject_mean_mae,
        "subject_mean_rmse": row.validation_subject_mean_rmse,
        "selected": row.alpha == fitted_states[row.evaluation_protocol]["selected_alpha"],
        "status": "fixed-grid validation selection",
    })

for row in performance_summary.itertuples(index=False):
    result_rows.append({
        "record_type": "performance",
        "evaluation_protocol": row.evaluation_protocol,
        "split": row.split,
        "model": row.model,
        "alpha": (
            fixed_alpha_by_protocol[row.evaluation_protocol]
            if row.model == "Ridge baseline (G+A+D+B)"
            else fitted_states[row.evaluation_protocol]["selected_alpha"]
        ),
        "subjects": row.subjects,
        "windows": row.windows,
        "subject_mean_mae": row.subject_mean_mae,
        "subject_mean_rmse": row.subject_mean_rmse,
        "pooled_mae": row.pooled_mae,
        "pooled_rmse": row.pooled_rmse,
        "status": "exploratory; identical common cohort",
    })

for row in gain_summary.itertuples(index=False):
    delta = row.delta_subject_mean_rmse
    if delta < -0.1:
        interpretation = "lower subject-mean RMSE; exploratory gain"
    elif delta > 0.1:
        interpretation = "higher subject-mean RMSE; negative result"
    else:
        interpretation = "negligible subject-mean RMSE difference"
    result_rows.append({
        "record_type": "paired_gain",
        "evaluation_protocol": row.evaluation_protocol,
        "split": row.split,
        "subjects": row.subjects,
        "windows": row.windows,
        "delta_subject_mean_mae": row.delta_subject_mean_mae,
        "delta_subject_mean_rmse": row.delta_subject_mean_rmse,
        "subjects_improved_mae": row.subjects_improved_mae,
        "subjects_improved_rmse": row.subjects_improved_rmse,
        "status": interpretation,
    })

microbiome_extension_results = pd.DataFrame(result_rows)
microbiome_extension_results["method_version"] = METHOD_VERSION
microbiome_extension_results["microbiome_semantics"] = "binary detection indicator; not abundance"
microbiome_extension_results["comparison_baseline"] = "Ridge (G+A+D+B), retrained on identical common cohort"
microbiome_extension_results["confirmation_scope"] = "exploratory extension after core test review"
microbiome_extension_results["preprocessing_scope"] = "fit on training subjects/training windows only"

results_path = EXT_DIR / "microbiome_extension_results.csv"
screening_path = EXT_DIR / "microbiome_feature_screening.csv"
predictions_path = EXT_DIR / "microbiome_extension_predictions.parquet"

microbiome_extension_results.to_csv(results_path, index=False)
microbiome_feature_screening.to_csv(screening_path, index=False)
microbiome_extension_predictions.to_parquet(predictions_path, index=False)

Microbiome coverage, screening stability and matched-cohort gains are recorded in `microbiome_extension_results.csv` and `microbiome_feature_screening.csv`. The gain labels use a descriptive 0.1 mg/dL convention, not a significance threshold; `delta_subject_mean_rmse` is the reported effect.


## Device and horizon sensitivity

Compare Dexcom at 30 minutes and Libre at 60 minutes with the primary task. Retain the 120-minute history, five-minute origin stride, source-continuity and full-support split rules. Each task builds its own labels/windows and fits preprocessing on its TRAIN data only. GlucoBench LR uses 24 lags; persistence is the no-fit reference.


In [8]:
# Define inputs for device and horizon sensitivity tasks.
sensitivity_paths = {
    "dynamic_aligned": ALIGNED_DIR / "dynamic_aligned_1min.parquet",
    "task_protocol": WINDOWS_DIR / "task_protocol.json",
    "window_rules": WINDOWS_DIR / "window_construction_rules.json",
    "split_rules": WINDOWS_DIR / "split_rules.json",
    "window_manifest": WINDOWS_DIR / "window_manifest.parquet",
    "fold_assignments": WINDOWS_DIR / "fold_assignments.parquet",
    "split_scope": DATA_QUALITY_DIR / "split_scope_v1.csv",
    "baseline_registry": BASELINE_DIR / "baseline_registry.json",
    "core_test_metrics": ART06 / "metrics_summary.csv",
}

task_protocol = json.loads(sensitivity_paths["task_protocol"].read_text(encoding="utf-8"))
window_rules = json.loads(sensitivity_paths["window_rules"].read_text(encoding="utf-8"))
split_rules = json.loads(sensitivity_paths["split_rules"].read_text(encoding="utf-8"))
baseline_registry = json.loads(sensitivity_paths["baseline_registry"].read_text(encoding="utf-8"))
primary = task_protocol["primary_task"]
candidate_ids = {item["task_id"] for item in task_protocol["sensitivity_candidates"]}
assert {"sensitivity_horizon_60min", "sensitivity_dexcom_target"} <= candidate_ids

HISTORY_MINUTES = int(primary["history_minutes"])
ORIGIN_STRIDE_MINUTES = int(primary["origin_stride_minutes"])
GRID_MINUTES = int(primary["grid_resolution_minutes"])
LR_IN_LEN = 24
LR_STEP_MINUTES = 5
LAG_OFFSETS = np.arange(-(LR_IN_LEN - 1) * LR_STEP_MINUTES, 1, LR_STEP_MINUTES)
assert HISTORY_MINUTES == 120 and len(LAG_OFFSETS) == LR_IN_LEN and LAG_OFFSETS[-1] == 0
for protocol in PROTOCOLS:
    fixed_lr = baseline_registry["models"]["formal baseline reproduction"]["glucobench_linear_regression"]["protocols"][protocol]
    assert fixed_lr["config"] == "in_len=24"
    assert fixed_lr["params"]["lag_step_minutes"] == LR_STEP_MINUTES
    assert fixed_lr["params"]["fit_intercept"] is True

TASKS = [
    {
        "sensitivity_id": "dexcom_target_30min",
        "task_id": "sensitivity_dexcom_30min",
        "target_device": "Dexcom",
        "target_field": "dexcom_gl",
        "history_field": "dexcom_gl",
        "forecast_horizon_minutes": 30,
        "changed_factor": "target_device",
        "comparison_task": "primary_libre_30min",
    },
    {
        "sensitivity_id": "libre_horizon_60min",
        "task_id": "sensitivity_libre_horizon60",
        "target_device": "Libre",
        "target_field": "libre_gl",
        "history_field": "libre_gl",
        "forecast_horizon_minutes": 60,
        "changed_factor": "forecast_horizon",
        "comparison_task": "primary_libre_30min",
    },
]
for task in TASKS:
    task.update({
        "target_unit": "mg/dL",
        "history_minutes": HISTORY_MINUTES,
        "origin_stride_minutes": ORIGIN_STRIDE_MINUTES,
        "grid_resolution_minutes": GRID_MINUTES,
        "history_interval": "[t-120min, t]",
        "prediction_type": "single_future_point_glucose_forecasting",
        "window_rule": window_rules["rule_version"],
        "split_rule": split_rules["version"],
        "baseline_model": "GlucoBench Linear Regression; in_len=24; training-only MinMaxScaler",
        "analysis_scope": "exploratory supplementary sensitivity",
    })

In [9]:
# Load the aligned glucose grid and construct each task's windows with the strict history, target and source-continuity rules.
aligned = pd.read_parquet(
    sensitivity_paths["dynamic_aligned"],
    columns=[
        "subject", "timestamp", "source_segment_id", "source_file", "source_row_index", "is_published_source_row",
        "is_inserted_grid_row", "libre_gl", "dexcom_gl", "libre_gl_value_origin", "dexcom_gl_value_origin",
    ],
).sort_values(["subject", "timestamp"]).reset_index(drop=True)
time_segments = pd.read_csv(ALIGNED_DIR / "time_segments.csv", parse_dates=["segment_start_time", "segment_end_time"])
EXCLUSION_STAGES = ["insufficient_history_extent", "target_outside_subject_extent", "disallowed_source_discontinuity",
                    "incomplete_published_primary_history", "target_unavailable"]


def construct_strict_task(task):
    """Construct one task's windows (src.windows.build_windows) with the lag values of the GlucoBench linear regression."""

    horizon = int(task["forecast_horizon_minutes"])
    field = task["history_field"]
    assert field == task["target_field"]
    _, flow_table, manifest, _ = build_windows(
        aligned, time_segments, task_id=task["task_id"], target_field=field, origin_field=f"{field}_value_origin",
        target_unit="mg/dL", grid_delta=pd.Timedelta(minutes=GRID_MINUTES), history_delta=pd.Timedelta(minutes=HISTORY_MINUTES),
        horizon_delta=pd.Timedelta(minutes=horizon), stride_delta=pd.Timedelta(minutes=ORIGIN_STRIDE_MINUTES))
    samples = manifest[["task_id", "subject", "origin_time", "history_start_time", "history_end_time", "target_time",
                        "segment_id", "source_file", "target_value"]].copy()
    samples.insert(0, "sensitivity_id", task["sensitivity_id"])
    samples["target_device"] = task["target_device"]
    samples["target_field"] = task["target_field"]
    samples["forecast_horizon_minutes"] = horizon
    samples["source_segment_id"] = samples["segment_id"].astype(int)
    lag_values = gather_lags(aligned, samples, LAG_OFFSETS, field)  # asserts 1-minute continuity and finite values
    samples = pd.concat([samples, pd.DataFrame(lag_values, columns=[f"g{offset}" for offset in LAG_OFFSETS])], axis=1)

    counts = flow_table.set_index("rule")
    stage_counts = ([("candidate_origins", counts.at["candidate_origins", "remaining_count"])]
                    + [(stage, counts.at[stage, "excluded_count"]) for stage in EXCLUSION_STAGES]
                    + [("eligible_windows", counts.at["eligible_windows", "input_count"])])
    flow = pd.DataFrame([{"sensitivity_id": task["sensitivity_id"], "stage": stage, "count": count} for stage, count in stage_counts])
    assert (samples["target_time"] > samples["origin_time"]).all()
    return samples, flow


In [10]:
# --- Assign strict temporal and subject-level splits to sensitivity windows ---

scope = pd.read_csv(sensitivity_paths["split_scope"])
scope = scope.rename(columns={"subject_id": "subject"})
for column in ["start_time", "end_time", "protocol_a_train_end_time", "protocol_a_val_end_time"]:
    scope[column] = pd.to_datetime(scope[column])
scope["protocol_b_group"] = scope["protocol_b_group"].replace({"val": "validation"})
assert set(scope["protocol_b_group"]) == {"train", "validation", "test"}


SAMPLE_KEY = ["subject", "origin_time", "task_id"]
WINDOW_COLUMNS = ["task_id", "subject", "origin_time", "history_start_time", "history_end_time", "target_time", "segment_id", "source_file"]


def split_task_samples(samples):
    """Assign every window to both evaluation protocols (src.windows.assign_splits); one row per window and protocol."""

    _, _, folds = assign_splits(samples[WINDOW_COLUMNS], scope, window_columns=WINDOW_COLUMNS, labels=["train", "validation", "test"],
                                protocol_a_name=PROTOCOL_A, protocol_b_name=PROTOCOL_B, sample_key=SAMPLE_KEY)
    assigned = samples.merge(folds[SAMPLE_KEY + ["evaluation_protocol", "split", "admitted_to_split"]],
                             on=SAMPLE_KEY, how="left", validate="one_to_many")
    # Boundary-crossing windows keep the label used in the saved coverage table.
    assigned["split"] = assigned["split"].replace({"excluded": "excluded_boundary"})
    return assigned


# --- Construct sensitivity samples and assign splits ---

sample_parts, flow_parts, assigned_parts = [], [], []
for task in TASKS:
    samples, flow = construct_strict_task(task)
    assigned = split_task_samples(samples)
    sample_parts.append(samples)
    flow_parts.append(flow)
    assigned_parts.append(assigned)

sensitivity_samples = pd.concat(sample_parts, ignore_index=True)
sensitivity_sample_flow = pd.concat(flow_parts, ignore_index=True)
sensitivity_assignments = pd.concat(assigned_parts, ignore_index=True)

coverage = (
    sensitivity_assignments.groupby(
        ["sensitivity_id", "task_id", "target_device", "forecast_horizon_minutes", "evaluation_protocol", "split"],
        observed=True,
    )
    .agg(subjects=("subject", "nunique"), windows=("subject", "size"))
    .reset_index()
)
core_counts = pd.DataFrame(baseline_registry["sample_keys"]["admitted_counts"])
core_counts = core_counts.rename(columns={"windows": "core_windows", "subjects": "core_subjects"})
coverage = coverage.merge(core_counts, on=["evaluation_protocol", "split"], how="left", validate="many_to_one")
coverage["delta_windows_vs_core"] = coverage["windows"] - coverage["core_windows"]
coverage["window_fraction_vs_core"] = coverage["windows"] / coverage["core_windows"]

In [11]:
# Evaluate persistence and fit linear-reference models for each sensitivity task.
LAG_COLUMNS = [f"g{offset}" for offset in LAG_OFFSETS]

prediction_parts = []
metric_rows = []
for task in TASKS:
    task_data = sensitivity_assignments.loc[
        sensitivity_assignments["sensitivity_id"].eq(task["sensitivity_id"])
        & sensitivity_assignments["admitted_to_split"]
    ].copy()
    for protocol in PROTOCOLS:
        protocol_data = task_data.loc[task_data["evaluation_protocol"].eq(protocol)].copy()
        train = protocol_data.loc[protocol_data["split"].eq("train")]
        assert len(train) and train[LAG_COLUMNS + ["target_value"]].notna().all().all()

        # The scaler is fitted on training lags and targets only, as in the baseline.
        lr_model, scaler = fit_glucobench(train, LAG_COLUMNS)
        protocol_data["persistence_prediction"] = protocol_data["g0"]
        protocol_data["glucobench_lr_prediction"] = predict_glucobench(lr_model, scaler, protocol_data, LAG_COLUMNS)
        assert protocol_data[["persistence_prediction", "glucobench_lr_prediction"]].notna().all().all()

        for split in ["train", "validation", "test"]:
            split_data = protocol_data.loc[protocol_data["split"].eq(split)]
            assert len(split_data)
            for model_id, prediction_column in [
                ("persistence", "persistence_prediction"),
                ("glucobench_linear_regression", "glucobench_lr_prediction"),
            ]:
                metric_rows.append({
                    "sensitivity_id": task["sensitivity_id"],
                    "task_id": task["task_id"],
                    "target_device": task["target_device"],
                    "forecast_horizon_minutes": task["forecast_horizon_minutes"],
                    "evaluation_protocol": protocol,
                    "split": split,
                    "model_id": model_id,
                    "config": "no parameters" if model_id == "persistence" else "in_len=24",
                    **subject_metric_summary(split_data, prediction_column)[0],
                })
        prediction_parts.append(protocol_data[[
            "sensitivity_id", "task_id", "target_device", "forecast_horizon_minutes",
            "evaluation_protocol", "split", "subject", "origin_time", "target_time", "target_value",
            "persistence_prediction", "glucobench_lr_prediction",
        ]])

sensitivity_predictions = pd.concat(prediction_parts, ignore_index=True)
sensitivity_metrics = pd.DataFrame(metric_rows)

In [12]:
# Primary TEST metrics are read only after all sensitivity tasks, fits, and predictions are fixed.
core_metrics = pd.read_csv(sensitivity_paths["core_test_metrics"])
core_metrics = core_metrics.loc[
    core_metrics["model_id"].isin(["persistence", "glucobench_linear_regression"]),
    ["evaluation_protocol", "model_id", "subject_equal_mae", "subject_equal_rmse", "n_subjects", "n_windows"],
].drop_duplicates(["evaluation_protocol", "model_id"])
assert len(core_metrics) == len(PROTOCOLS) * 2

test_metrics = sensitivity_metrics.loc[sensitivity_metrics["split"].eq("test")].copy()
test_comparison = test_metrics.merge(
    core_metrics,
    on=["evaluation_protocol", "model_id"],
    how="left",
    validate="many_to_one",
    suffixes=("_sensitivity", "_core_libre30"),
)
test_comparison["delta_subject_mean_mae_vs_core_libre30"] = (
    test_comparison["subject_mean_mae"] - test_comparison["subject_equal_mae"]
)
test_comparison["delta_subject_mean_rmse_vs_core_libre30"] = (
    test_comparison["subject_mean_rmse"] - test_comparison["subject_equal_rmse"]
)

within_task = sensitivity_metrics.pivot_table(
    index=["sensitivity_id", "task_id", "target_device", "forecast_horizon_minutes", "evaluation_protocol", "split", "subjects", "windows"],
    columns="model_id",
    values=["subject_mean_mae", "subject_mean_rmse"],
).reset_index()
within_task.columns = ["_".join([str(value) for value in column if str(value)]) if isinstance(column, tuple) else column for column in within_task.columns]
within_task["delta_lr_vs_persistence_mae"] = (
    within_task["subject_mean_mae_glucobench_linear_regression"] - within_task["subject_mean_mae_persistence"]
)
within_task["delta_lr_vs_persistence_rmse"] = (
    within_task["subject_mean_rmse_glucobench_linear_regression"] - within_task["subject_mean_rmse_persistence"]
)


In [13]:
# Assemble and persist device and horizon sensitivity artifacts.
result_rows = []
task_lookup = {task["sensitivity_id"]: task for task in TASKS}
for task in TASKS:
    result_rows.append({
        "record_type": "task_definition",
        **task,
        "status": "evaluated_once; no adaptive expansion",
    })

for row in sensitivity_sample_flow.itertuples(index=False):
    task = task_lookup[row.sensitivity_id]
    result_rows.append({
        "record_type": "sample_flow",
        "sensitivity_id": row.sensitivity_id,
        "task_id": task["task_id"],
        "target_device": task["target_device"],
        "forecast_horizon_minutes": task["forecast_horizon_minutes"],
        "stage": row.stage,
        "count": row.count,
        "status": "sequential first-failure flow",
    })

for row in coverage.itertuples(index=False):
    result_rows.append({
        "record_type": "coverage",
        "sensitivity_id": row.sensitivity_id,
        "task_id": row.task_id,
        "target_device": row.target_device,
        "forecast_horizon_minutes": row.forecast_horizon_minutes,
        "evaluation_protocol": row.evaluation_protocol,
        "split": row.split,
        "subjects": row.subjects,
        "windows": row.windows,
        "core_windows": row.core_windows,
        "delta_windows_vs_core": row.delta_windows_vs_core,
        "window_fraction_vs_core": row.window_fraction_vs_core,
        "status": "strict task-specific coverage",
    })

for row in sensitivity_metrics.itertuples(index=False):
    result_rows.append({
        "record_type": "metric",
        "sensitivity_id": row.sensitivity_id,
        "task_id": row.task_id,
        "target_device": row.target_device,
        "forecast_horizon_minutes": row.forecast_horizon_minutes,
        "evaluation_protocol": row.evaluation_protocol,
        "split": row.split,
        "model_id": row.model_id,
        "config": row.config,
        "subjects": row.subjects,
        "windows": row.windows,
        "subject_mean_mae": row.subject_mean_mae,
        "subject_mean_rmse": row.subject_mean_rmse,
        "pooled_mae": row.pooled_mae,
        "pooled_rmse": row.pooled_rmse,
        "status": "exploratory supplementary metric",
    })

for row in test_comparison.itertuples(index=False):
    result_rows.append({
        "record_type": "core_test_comparison",
        "sensitivity_id": row.sensitivity_id,
        "task_id": row.task_id,
        "target_device": row.target_device,
        "forecast_horizon_minutes": row.forecast_horizon_minutes,
        "evaluation_protocol": row.evaluation_protocol,
        "split": "test",
        "model_id": row.model_id,
        "config": row.config,
        "subjects": row.subjects,
        "windows": row.windows,
        "subject_mean_mae": row.subject_mean_mae,
        "subject_mean_rmse": row.subject_mean_rmse,
        "core_libre30_subject_mean_mae": row.subject_equal_mae,
        "core_libre30_subject_mean_rmse": row.subject_equal_rmse,
        "delta_subject_mean_mae_vs_core_libre30": row.delta_subject_mean_mae_vs_core_libre30,
        "delta_subject_mean_rmse_vs_core_libre30": row.delta_subject_mean_rmse_vs_core_libre30,
        "status": "difference conflates target and sample-coverage changes",
    })

for row in within_task.itertuples(index=False):
    result_rows.append({
        "record_type": "within_task_model_comparison",
        "sensitivity_id": row.sensitivity_id,
        "task_id": row.task_id,
        "target_device": row.target_device,
        "forecast_horizon_minutes": row.forecast_horizon_minutes,
        "evaluation_protocol": row.evaluation_protocol,
        "split": row.split,
        "subjects": row.subjects,
        "windows": row.windows,
        "delta_lr_vs_persistence_mae": row.delta_lr_vs_persistence_mae,
        "delta_lr_vs_persistence_rmse": row.delta_lr_vs_persistence_rmse,
        "status": "same-task same-sample comparison",
    })

sensitivity_results = pd.DataFrame(result_rows)
sensitivity_results["method_version"] = "device_horizon_sensitivity_v1"

sensitivity_results["analysis_scope"] = "exploratory supplementary robustness analysis"
sensitivity_results["leakage_policy"] = "strict history through origin; task-specific target; training-only scaling and fitting"

protocol_artifact = {
    "method_version": "device_horizon_sensitivity_v1",
    "analysis_scope": "exploratory supplementary robustness analysis",
    "tasks": TASKS,
    "models": {
        "persistence": "prediction equals the task device value at origin",
        "glucobench_linear_regression": {
            "config": "in_len=24",
            "lag_offsets_minutes": LAG_OFFSETS.tolist(),
            "fit_intercept": True,
            "preprocessing": "one MinMaxScaler per task and evaluation protocol, fitted on training lags and targets only",
        },
    },
    "not_evaluated": {
        "alternative_history": "no alternative history length evaluated",
        "alternative_grid": "no alternative grid evaluated",
        "alternative_split_or_cv": "no additional split/CV variant evaluated",

    },
}

results_path = EXT_DIR / "sensitivity_results.csv"
predictions_path = EXT_DIR / "sensitivity_predictions.parquet"
protocol_path = EXT_DIR / "sensitivity_task_protocols.json"
sensitivity_results.to_csv(results_path, index=False)
sensitivity_predictions.to_parquet(predictions_path, index=False)
_ = protocol_path.write_text(json.dumps(protocol_artifact, indent=2), encoding="utf-8")


The table reports task-specific coverage and GlucoBench LR/persistence metrics. These supplementary results reuse TEST and are exploratory.


## PPGR evidence review and reconstruction protocol

### PPGR reconstruction basis

Reconstruct the published 2-hour breakfast AUC/iAUC task with 19 predictors and LOSO. XGBoost uses 80 trees, depth 1, learning rate 0.2, L1 = 1 and L2 = 0. The author notebook (`parse_data.ipynb`, commit `3d0c9b7`) supplies integration, baseline, HOMA-IR and train-only scaling rules; unresolved paper/code differences have explicit choices in the table below.

Source: Das et al., *Scientific Data* 12:1557 (2025), doi:10.1038/s41597-025-05851-7. Results are labeled `evidence_bound_partial_reconstruction`.


In [14]:
# Current-dataset evidence: canonical schemas, the raw Bio header and the two source dictionaries.
schema_ppgr = {name: pq.read_schema(PROJECT_ROOT / "results" / "aligned_data" / f"{file}.parquet").names
               for name, file in [("bio", "bio_canonical"), ("food", "food_entries"), ("aligned", "dynamic_aligned_1min")]}
bio_header_ppgr = pd.read_csv(DATA_ROOT / "CGMacros" / "bio.csv", nrows=0).columns.tolist()
dictionary_ppgr = {name: pd.read_csv(DATA_ROOT / file).assign(Field=lambda d: d["Field"].str.strip()).set_index("Field")
                   for name, file in [("dict_bio", "DataDictionary_Bio.csv"), ("dict_dyn", "DataDictionary_CGMacros-00X.csv")]}
assert not [c for c in bio_header_ppgr if "homa" in c.lower()]

In [15]:
# The adopted rules below rely on these canonical-table fields and data-dictionary statements; both are checked against the current dataset.
FIDELITY_PPGR = "evidence_bound_partial_reconstruction"
REQUIRED_FIELDS = {
    "food": ["meal_type", "is_meal_anchor", "meal_type_raw", "carbs", "protein", "fat", "fiber", "timestamp", "source_row_index", "image_path", "amount_consumed", "nutrition_adjustment_applied", "nutrition_values_are_raw", "subject"],
    "bio": ["age", "gender", "bmi", "hba1c_lab_raw", "fasting_glucose_lab_mg_dl", "fasting_insulin_raw", "triglycerides_mg_dl", "cholesterol_mg_dl", "hdl_mg_dl", "non_hdl_mg_dl", "ldl_calculated_mg_dl", "vldl_calculated_mg_dl", "cholesterol_hdl_ratio_raw", "body_weight_lb", "height_in", "subject"],
    "aligned": ["libre_gl", "is_inserted_grid_row", "libre_gl_value_origin", "timestamp", "source_row_index", "gap_id"],
}
DICTIONARY_STATEMENTS = {
    "dict_dyn": [("Meal Type", "Breakfast"), ("Carbs", "grams"), ("Amount Consumed", "%"), ("Libre GL", "mg/dL")],
    "dict_bio": [("Insulin", "mcU/mL"), ("Fasting GLU - PDL (Lab)", "mg/dL"), ("Gender", "M"), ("Body weight", "pounds"), ("Height", "inches"),
                 ("A1c PDL (Lab)", "mmol/mol"), ("Cho/HDL Ratio", "400"), ("LDL (Cal)", "800"), ("VLDL (Cal)", "400")],
}
missing_fields = [f"{table}:{field}" for table, fields in REQUIRED_FIELDS.items() for field in fields if field not in schema_ppgr[table]]
missing_statements = [f"{table}:{field}~{text}" for table, statements in DICTIONARY_STATEMENTS.items() for field, text in statements
                      if text.lower() not in f"{dictionary_ppgr[table].loc[field, 'Range']} {dictionary_ppgr[table].loc[field, 'Description']}".lower()]
assert not missing_fields and not missing_statements, (missing_fields, missing_statements)
print(f"Current dataset supports the reconstruction: {sum(map(len, REQUIRED_FIELDS.values()))} table fields and "
      f"{sum(map(len, DICTIONARY_STATEMENTS.values()))} dictionary statements confirmed")

Current dataset supports the reconstruction: 36 table fields and 13 dictionary statements confirmed


### Evidence conflicts and implementation risks

The gaps below remain in the public evidence (the four nutrient-energy conversions share one row); each has a fixed resolution for this reconstruction.

| Element | Adopted rule | Sensitivity |
|---|---|---|
| Breakfast selection | Meal-anchor food entry with Meal Type `breakfast` (case-insensitive) and released Libre values at all nine timestamps anchor + 0, 15,..., 120 min within one continuous recording segment. | none |
| iAUC > 0 exclusion | No iAUC-based exclusion (the author code additionally excludes non-positive iAUC, which also shrinks the AUC sample). | required |
| First-breakfast exclusion | No first-breakfast exclusion. | not run |
| Carbohydrate, protein, fat, fiber energy | Single conversion: x4, x4, x9, x2 (the author code applies the carbohydrate, protein and fiber factors twice, i.e. effectively x16, x16, x4). | required (x16, x16, x4 refits check the scaling invariance) |
| Amount Consumed | Released nutrient values are used unchanged. | not run |
| Baseline glucose | Released Libre value at the meal-anchor timestamp; used as a predictor and as the iAUC reference. | not run |
| Endpoint geometry | Libre values at anchor + 0, 15,..., 120 min located by timestamp; meals lacking a timestamp or crossing a recording gap are excluded. | none |
| AUC integration | Trapezoidal integral in mg/dL*min. | none |
| iAUC below baseline | Area above the baseline over 0-120 min: fully-above intervals contribute the trapezoid, crossing intervals the triangle up to the crossing, fully-below intervals 0. | not run |
| Area units | mg/dL*min; values divided by 60 are labeled mg/dL*h and used only for explicit hour-based comparison. | none |
| Missing glucose | A meal with a missing value at any of the nine timestamps is excluded; target glucose is never imputed. | none |
| HOMA-IR | \(\mathrm{HOMA\! -\! IR}=\mathrm{insulin}\,(\mu\mathrm{U/mL})\times\mathrm{glucose}\,(\mathrm{mg/dL})/405\), joined by subject ID; insulin text parsed to its numeric part, `(low)` entries flagged with the value retained. | none |
| Gender | M = 1, F = -1; any other or missing value is a data issue for that subject and is not encoded. | none |
| BMI | Derived from weight and height per subject. | diagnostic (agreement with the supplied BMI) |
| HbA1c units | Raw values are used without conversion and read as percent; the dictionary conflict is recorded. | none |
| CHO/HDL ratio | Used as a dimensionless raw ratio. | none |
| Dictionary-flagged errors | LDL (Cal) = 800 and VLDL (Cal) = 400 are set to missing; other extreme values are kept. | none |
| Missing predictors | Complete case over the 19 predictors; excluded subjects and reasons are reported. | not run |
| Subject-level join | Joined to meals by subject ID; positional assignment is prohibited (the author code assigns static values by position, checked by the positional-versus-ID diagnostic). | diagnostic (positional versus ID assignment) |
| Model settings | XGBoost 1.6.2 with the five paper values; every other parameter at the 1.6.2 default, recorded before fitting and not changed afterwards; fitting does not start if 1.6.2 cannot run in a recorded environment. | none |
| Evaluation cohort | LOSO over every subject with at least one eligible breakfast and complete predictors; the actual cohort size is reported and 44/1 is a reference only. | none |
| Software environment | Separately recorded environment built from `requirements-ppgr.txt`; the versions actually used are recorded with the results. | none |
| Random seed | `random_state = 0`. | not run |
| Source version | Commit `3d0c9b7` is the implementation reference; results are labeled `evidence_bound_partial_reconstruction`, never a faithful reproduction. | none |


### Sensitivities and diagnostics

Compare the author-code iAUC \(>0\) exclusion and scaling invariance for carbohydrate, protein and fiber. Diagnostics check positional versus subject-ID alignment and derived versus supplied BMI. Other variants are not evaluated. XGBoost 1.6.2 is required.


## Evidence-bound PPGR reproduction

This section executes the PPGR reconstruction defined by the protocol above without reinterpreting any adopted rule.

In [16]:
# Reconstruction constants fixed by the protocol above.
NUTRIENT_FACTORS = {"carbs": 4, "protein": 4, "fat": 9, "fiber": 2}
AUTHOR_FACTORS = {"carbs": 16, "protein": 16, "fiber": 4}
OFFSETS = list(range(0, 121, 15))
XGB_PARAMS = {"n_estimators": 80, "max_depth": 1, "learning_rate": 0.2, "reg_alpha": 1, "reg_lambda": 0, "random_state": 0}
XGB_VERSION = "1.6.2"
ERROR_VALUES = {"ldl_calculated_mg_dl": 800, "vldl_calculated_mg_dl": 400}

In [17]:
# Model fitting runs only in the separately recorded reproduction environment with XGBoost 1.6.2.
PPGR_PYTHON = Path(os.environ.get("CGMACROS_PPGR_PYTHON", PROJECT_ROOT / ".venv-ppgr" / "Scripts" / "python.exe"))

probe = ("import json, platform, sys, xgboost, sklearn, pandas, numpy, scipy\n"
         "print(json.dumps({'python': platform.python_version(), 'implementation': platform.python_implementation(),"
         " 'platform': platform.platform(), 'xgboost': xgboost.__version__, 'scikit-learn': sklearn.__version__,"
         " 'pandas': pandas.__version__, 'numpy': numpy.__version__, 'scipy': scipy.__version__}))")
if not PPGR_PYTHON.is_file():
    raise FileNotFoundError(f"Reproduction interpreter not found: {PPGR_PYTHON}. Set CGMACROS_PPGR_PYTHON to the Python 3.9 executable from requirements-ppgr.txt.")
environment = json.loads(subprocess.run([str(PPGR_PYTHON), "-c", probe], capture_output=True, text=True, check=True).stdout)
if environment["xgboost"] != XGB_VERSION:
    raise ValueError(f"XGBoost {XGB_VERSION} required; found {environment['xgboost']}.")


The full anchor \(+\,[0, 120]\) min interval must remain within one continuous source recording segment and consist entirely of published source rows; Libre values are read at the nine 15-min sampling points \(+\,0, 15, \ldots, 120\) min. Baseline is the anchor value; AUC is the trapezoidal integral and iAUC the positive area above baseline, both in mg/dL*min.

In [18]:
# --- Construct eligible breakfast targets and PPGR outcomes ---

food = pd.read_parquet(PROJECT_ROOT / "results" / "aligned_data" / "food_entries.parquet")
grid = pd.read_parquet(PROJECT_ROOT / "results" / "aligned_data" / "dynamic_aligned_1min.parquet",
                       columns=["subject", "timestamp", "libre_gl", "source_segment_id", "is_published_source_row"])
grid = grid.set_index(["subject", "timestamp"]).sort_index()
# Filtered on meal_type_raw only: breakfast rows are meal anchors by construction, because a populated
# meal_type satisfies the is_meal_anchor definition in data alignment (food_entries["is_meal_anchor"] =
# meal_type.notna() | has_any_core_nutrition | amount_consumed.notna()), so this filter already matches
# the "meal-anchor food entry" protocol.
breakfasts = food.loc[food["meal_type_raw"].str.lower().eq("breakfast")].sort_values(["subject", "timestamp"]).reset_index(drop=True)


# --- Define positive incremental AUC on the fixed meal grid ---

def iauc_positive(values, dx=15.0):
    """Area of the piecewise-linear curve above values[0]; crossings at the linear intersection, fully-below intervals 0."""
    d = np.asarray(values, dtype=float) - values[0]
    area = 0.0
    for a, b in zip(d[:-1], d[1:]):
        if a >= 0 and b >= 0:
            area += (a + b) / 2 * dx
        elif a >= 0 > b:
            area += a * (a / (a - b)) * dx / 2
        elif b >= 0 > a:
            area += b * (b / (b - a)) * dx / 2
    return area


# --- Build one PPGR manifest row per breakfast entry ---

rows = []
for meal in breakfasts.itertuples():
    t0 = meal.timestamp
    window = grid.loc[meal.subject].loc[t0:t0 + pd.Timedelta(minutes=120)]
    samples = window["libre_gl"].reindex([t0 + pd.Timedelta(minutes=m) for m in OFFSETS]).to_numpy(dtype=float)
    continuous = (len(window) == 121 and window["is_published_source_row"].all()
                  and window["source_segment_id"].notna().all() and window["source_segment_id"].nunique() == 1)
    reason = ("window_crosses_recording_gap_or_incomplete" if not continuous
              else "missing_glucose_sample" if np.isnan(samples).any() else "")
    row = {"meal_id": meal.food_entry_id, "subject": meal.subject, "anchor_time": t0, "meal_type_raw": meal.meal_type_raw,
           "meal_exclusion_reason": reason}
    row.update({f"glucose_{m:03d}min": v for m, v in zip(OFFSETS, samples)})
    row["baseline_glucose"] = samples[0]
    row["auc_mg_dl_min"] = float(np.sum((samples[1:] + samples[:-1]) / 2 * 15)) if not reason else np.nan
    row["iauc_mg_dl_min"] = iauc_positive(samples) if not reason else np.nan
    for nutrient in NUTRIENT_FACTORS:
        row[f"{nutrient}_g"] = getattr(meal, nutrient)
    rows.append(row)
manifest = pd.DataFrame(rows)
eligible_meals = manifest["meal_exclusion_reason"].eq("")
print(f"Breakfast entries: {len(manifest)} from {manifest['subject'].nunique()} subjects; eligible meals: {int(eligible_meals.sum())}")

Breakfast entries: 436 from 45 subjects; eligible meals: 418


In [19]:
# Subject-level predictors joined by subject ID, nutrient energy and the complete-case cohort.
bio = pd.read_parquet(PROJECT_ROOT / "results" / "aligned_data" / "bio_canonical.parquet")
bio = bio.loc[bio["in_dynamic_cohort"]].set_index("subject")
static = pd.DataFrame(index=bio.index)
static["age"] = bio["age"].astype(float)
static["gender"] = bio["gender"].map({"M": 1.0, "F": -1.0})
static["bmi"] = bio["body_weight_lb"] * 0.453592 / (bio["height_in"] * 0.0254) ** 2
static["hba1c"] = bio["hba1c_lab_raw"]
insulin_text = bio["fasting_insulin_raw"].astype(str)
static["fasting_insulin"] = pd.to_numeric(insulin_text.str.extract(r"([-+]?\d*\.?\d+)")[0], errors="coerce")
static["homa_ir"] = static["fasting_insulin"] * bio["fasting_glucose_lab_mg_dl"] / 405
static["triglycerides"] = bio["triglycerides_mg_dl"].astype(float)
static["cholesterol"] = bio["cholesterol_mg_dl"].astype(float)
static["hdl"] = bio["hdl_mg_dl"].astype(float)
static["non_hdl"] = bio["non_hdl_mg_dl"].astype(float)
static["ldl"] = bio["ldl_calculated_mg_dl"].astype(float).mask(bio["ldl_calculated_mg_dl"].eq(ERROR_VALUES["ldl_calculated_mg_dl"]))
static["vldl"] = bio["vldl_calculated_mg_dl"].astype(float).mask(bio["vldl_calculated_mg_dl"].eq(ERROR_VALUES["vldl_calculated_mg_dl"]))
static["cholesterol_hdl_ratio"] = bio["cholesterol_hdl_ratio_raw"]
static["fasting_glucose"] = bio["fasting_glucose_lab_mg_dl"].astype(float)

PREDICTORS = ["carbs_energy", "protein_energy", "fat_energy", "fiber_energy", "baseline_glucose", "age", "gender", "bmi", "hba1c",
              "homa_ir", "fasting_insulin", "triglycerides", "cholesterol", "hdl", "non_hdl", "ldl", "vldl",
              "cholesterol_hdl_ratio", "fasting_glucose"]
assert len(PREDICTORS) == 19
for nutrient, factor in NUTRIENT_FACTORS.items():
    manifest[f"{nutrient}_energy"] = manifest[f"{nutrient}_g"] * factor
manifest = manifest.join(static, on="subject")
missing_static = static.isna()
subject_reason = missing_static.apply(lambda r: "missing_predictor: " + ", ".join(r.index[r]) if r.any() else "", axis=1)
manifest["subject_exclusion_reason"] = manifest["subject"].map(subject_reason).fillna("no_bio_record")
meal_ok = manifest["meal_exclusion_reason"].eq("") & manifest["subject_exclusion_reason"].eq("") & manifest[PREDICTORS].notna().all(axis=1)
manifest["in_primary_reconstruction"] = meal_ok
manifest["in_iauc_positive_sensitivity"] = meal_ok & manifest["iauc_mg_dl_min"].gt(0)
manifest["fidelity_label"] = FIDELITY_PPGR
cohort = manifest.loc[meal_ok]
print(f"Primary cohort: {len(cohort)} meals from {cohort['subject'].nunique()} subjects "
      f"(meals per subject {cohort.groupby('subject').size().min()}-{cohort.groupby('subject').size().max()})")
iauc_positive = manifest.loc[manifest["in_iauc_positive_sensitivity"]]
print(f"iAUC > 0 sensitivity sample: {len(iauc_positive)} meals from {iauc_positive['subject'].nunique()} subjects")

Primary cohort: 409 meals from 43 subjects (meals per subject 7-10)
iAUC > 0 sensitivity sample: 403 meals from 43 subjects


Each target is fitted with XGBoost 1.6.2 (80 trees, depth 1, learning rate 0.2, \(L_1 = 1\), \(L_2 = 0\), seed 0, other settings at their defaults) after a StandardScaler fitted on the training subjects of each fold.

In [20]:
# LOSO fits in the reproduction environment: train-only StandardScaler and XGBoost 1.6.2 per fold and target.
AnalyzeS = {"primary": (cohort, {}), "iauc_positive_exclusion": (iauc_positive, {})}
AnalyzeS.update({f"{nutrient}_factor_x{factor}": (cohort, {nutrient: factor}) for nutrient, factor in AUTHOR_FACTORS.items()})
RUNNER = r'''
import json, sys
import pandas as pd
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
data = pd.read_csv(sys.argv[1]); cfg = json.loads(open(sys.argv[2]).read()); out = []
for (analysis, target), part in data.groupby(["analysis", "target"], sort=False):
    for subject in sorted(part["subject"].unique()):
        train, test = part[part["subject"] != subject], part[part["subject"] == subject]
        scaler = StandardScaler().fit(train[cfg["predictors"]])
        model = XGBRegressor(**cfg["params"]).fit(scaler.transform(train[cfg["predictors"]]), train["y"])
        pred = model.predict(scaler.transform(test[cfg["predictors"]]))
        out.append(pd.DataFrame({"analysis": analysis, "target": target, "meal_id": test["meal_id"].values,
                                 "subject": subject, "y_true": test["y"].values, "y_pred": pred.astype(float)}))
pd.concat(out).to_csv(sys.argv[3], index=False, float_format="%.10g")
json.dump({"xgb_params": model.get_xgb_params(), "sklearn_params": model.get_params(), "config": json.loads(model.get_booster().save_config())},
          open(sys.argv[4], "w"), default=str)
'''
frames = []
for analysis, (frame, factors) in AnalyzeS.items():
    x = frame[["meal_id", "subject"] + PREDICTORS].copy()
    for nutrient, factor in factors.items():
        x[f"{nutrient}_energy"] = frame[f"{nutrient}_g"] * factor
    for target, column in [("AUC", "auc_mg_dl_min"), ("iAUC", "iauc_mg_dl_min")]:
        frames.append(x.assign(analysis=analysis, target=target, y=frame[column].to_numpy()))
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    pd.concat(frames).to_csv(tmp / "input.csv", index=False, float_format="%.10g")
    (tmp / "config.json").write_text(json.dumps({"predictors": PREDICTORS, "params": XGB_PARAMS}))
    (tmp / "runner.py").write_text(RUNNER)
    subprocess.run([str(PPGR_PYTHON), str(tmp / "runner.py"), str(tmp / "input.csv"), str(tmp / "config.json"),
                    str(tmp / "pred.csv"), str(tmp / "model.json")], check=True, capture_output=True, text=True)
    predictions = pd.read_csv(tmp / "pred.csv")
    model_record = json.loads((tmp / "model.json").read_text())
predictions["fidelity_label"] = FIDELITY_PPGR
assert set(predictions["analysis"]) == set(AnalyzeS)

In [21]:
# Pooled Pearson correlation over all held-out meals; the paper's values are descriptive references only.
PAPER_R = {"AUC": 0.89, "iAUC": 0.64}
ANALYSIS_ELEMENT = {"iauc_positive_exclusion": "sample_exclusion_positive_iauc", "carbs_factor_x16": "carbohydrate_conversion",
                    "protein_factor_x16": "protein_conversion", "fiber_factor_x4": "fiber_conversion"}
result_rows = []
for (analysis, target), part in predictions.groupby(["analysis", "target"], sort=False):
    r, p = pearsonr(part["y_pred"], part["y_true"])
    kind = "primary" if analysis == "primary" else "selected_required"
    element_id = ANALYSIS_ELEMENT.get(analysis, "")
    result_rows.append({"record_type": kind, "analysis": analysis, "element_id": element_id, "target": target, "n_meals": len(part),
                        "n_subjects": part["subject"].nunique(), "pearson_r": r, "p_value": p,
                        "paper_reported_r": PAPER_R[target], "r_minus_paper_reported": r - PAPER_R[target]})
results = pd.DataFrame(result_rows)

# Scaling invariance: predictions with author-code factors compared with the primary predictions on identical meals.
primary_pred = predictions.loc[predictions["analysis"].eq("primary")].set_index(["target", "meal_id"])["y_pred"]
invariance = {}
for analysis in [a for a in AnalyzeS if "_factor_x" in a]:
    variant = predictions.loc[predictions["analysis"].eq(analysis)].set_index(["target", "meal_id"])["y_pred"]
    for target in ["AUC", "iAUC"]:
        invariance[(analysis, target)] = float((variant.loc[target] - primary_pred.loc[target]).abs().max())
results["max_abs_prediction_difference_vs_primary"] = [invariance.get((a, t), np.nan) for a, t in zip(results["analysis"], results["target"])]
results["fidelity_label"] = FIDELITY_PPGR
display(results.round({"pearson_r": 4, "r_minus_paper_reported": 4, "max_abs_prediction_difference_vs_primary": 6})
        .assign(p_value=results["p_value"].map("{:.2e}".format)))

,record_type,analysis,element_id,target,n_meals,n_subjects,pearson_r,p_value,paper_reported_r,r_minus_paper_reported,max_abs_prediction_difference_vs_primary,fidelity_label
0,primary,primary,,AUC,409,43,0.8455,5.92e-113,0.89,-0.0445,NaN,evidence_bound_partial_reconstruction
1,primary,primary,,iAUC,409,43,0.6013,1.43e-41,0.64,-0.0387,NaN,evidence_bound_partial_reconstruction
2,selected_required,iauc_positive_exclusion,sample_exclusion_positive_iauc,AUC,403,43,0.8675,1.14e-123,0.89,-0.0225,NaN,evidence_bound_partial_reconstruction
3,selected_required,iauc_positive_exclusion,sample_exclusion_positive_iauc,iAUC,403,43,0.6021,4.18e-41,0.64,-0.0379,NaN,evidence_bound_partial_reconstruction
4,selected_required,carbs_factor_x16,carbohydrate_conversion,AUC,409,43,0.8455,5.92e-113,0.89,-0.0445,0.0,evidence_bound_partial_reconstruction
5,selected_required,carbs_factor_x16,carbohydrate_conversion,iAUC,409,43,0.6013,1.43e-41,0.64,-0.0387,0.0,evidence_bound_partial_reconstruction
6,selected_required,protein_factor_x16,protein_conversion,AUC,409,43,0.8455,5.92e-113,0.89,-0.0445,0.0,evidence_bound_partial_reconstruction
7,selected_required,protein_factor_x16,protein_conversion,iAUC,409,43,0.6013,1.43e-41,0.64,-0.0387,0.0,evidence_bound_partial_reconstruction
8,selected_required,fiber_factor_x4,fiber_conversion,AUC,409,43,0.8455,5.92e-113,0.89,-0.0445,0.0,evidence_bound_partial_reconstruction
9,selected_required,fiber_factor_x4,fiber_conversion,iAUC,409,43,0.6013,1.43e-41,0.64,-0.0387,0.0,evidence_bound_partial_reconstruction


Diagnostics: the author code's positional assignment of subject-level values, and agreement between derived and supplied BMI. Neither changes the reconstruction.

In [22]:
# Diagnostics only (subject-level join and BMI implementation); neither changes the reconstruction.
raw_bio = pd.read_csv(DATA_ROOT / "CGMacros" / "bio.csv")
POSITIONAL_COLUMNS = ["A1c PDL (Lab)", "Fasting GLU - PDL (Lab)", "Insulin ", "Triglycerides", "Cholesterol", "HDL", "Non HDL ",
                      "LDL (Cal)", "VLDL (Cal)", "Cho/HDL Ratio", "Body weight ", "Height "]
diagnostic_rows = []
for label, subjects in [("primary", sorted(cohort["subject"].unique())), ("iauc_positive_exclusion", sorted(iauc_positive["subject"].unique()))]:
    # Author-code alignment: column-wise dropna, then the i-th value is given to the i-th subject in sorted order.
    mismatched = set()
    for column in POSITIONAL_COLUMNS + ["Age", "Gender"]:
        values = raw_bio[column].dropna().to_numpy() if column in POSITIONAL_COLUMNS else raw_bio[column].to_numpy()
        keyed = raw_bio.set_index("subject")[column]
        for i, subject in enumerate(subjects):
            if i >= len(values) or str(values[i]) != str(keyed.loc[subject]):
                mismatched.add(subject)
    diagnostic_rows.append({"record_type": "diagnostic_only", "analysis": f"static_feature_join_positional_order_{label}", "element_id": "static_feature_join",
                            "n_subjects": len(subjects), "value": len(mismatched),
                            "detail": f"subjects whose positionally assigned static values differ from subject-ID values: {sorted(int(s) for s in mismatched)}"})
supplied = bio["bmi"]
for label, subjects in [("all_subjects", static.index), ("primary_cohort", sorted(cohort["subject"].unique()))]:
    diff = (static.loc[subjects, "bmi"] - supplied.loc[subjects]).abs()
    diagnostic_rows.append({"record_type": "diagnostic_only", "analysis": f"bmi_implementation_agreement_{label}", "element_id": "bmi_implementation",
                            "n_subjects": len(subjects), "value": float(diff.max()),
                            "detail": f"derived vs supplied BMI: max abs difference {diff.max():.4f}, mean {diff.mean():.4f}, "
                                      f"Pearson r {pearsonr(static.loc[subjects, 'bmi'], supplied.loc[subjects])[0]:.6f}"})
diagnostics = pd.DataFrame(diagnostic_rows).assign(fidelity_label=FIDELITY_PPGR)

In [23]:
# Persist the manifest, predictions and results (with environment and model settings); check that the primary analysis's predicted meal IDs match the eligible cohort exactly.
reproduction_results = pd.concat([results, diagnostics], ignore_index=True)
reproduction_results["reproduction_environment"] = json.dumps(environment, sort_keys=True)
reproduction_results["xgb_params"] = json.dumps(model_record["xgb_params"], sort_keys=True, default=str)
outputs = {"ppgr_sample_manifest.csv": manifest, "ppgr_predictions.csv": predictions, "ppgr_reproduction_results.csv": reproduction_results}
for name, frame in outputs.items():
    frame.to_csv(EXT_DIR / name, index=False, encoding="utf-8")
assert set(predictions.loc[predictions["analysis"].eq("primary"), "meal_id"]) == set(cohort["meal_id"])

Cohort construction, the primary reconstruction, the `selected_required` sensitivities (author-code iAUC \(> 0\) exclusion, nutrient-factor scaling invariance) and both diagnostics are recorded in the PPGR result artifacts above. The reported correlations are descriptive references only; the results are an evidence-bound partial reconstruction, not a faithful reproduction.

## Compact sequence models

### GRU model

A single-layer GRU summarizes the 25-value Libre history over \([t-120,t]\) at five-minute steps. Use hidden size 32, residual output on the last value, Adam (learning rate 0.001, batch 512), at most 60 epochs, validation early stopping with patience 6, and seeds 0-4. Normalization uses TRAIN only; TEST does not guide stopping or selection. Compare baselines on identical windows. Reuse of the primary TEST partition makes these results exploratory.


In [24]:
import torch
from src.seq_models import CompactGRU, CompactTCN

# GRU configuration and five-seed training budget.
CONFIG = {
    "extension_type": "compact_sequence_model",
    "architecture": "GRU",
    "input": "25 released Libre values at t-120, t-115, ..., t (5-min steps), standardized with training-split statistics",
    "output": "g0 + residual; the residual (target - g0) is standardized with training-split statistics",
    "hidden_size": 32, "num_layers": 1, "dropout": 0.0, "head": "Linear(32, 1) on the last hidden state",
    "loss": "mean squared error on the standardized residual", "optimizer": "Adam", "learning_rate": 1e-3, "weight_decay": 0.0,
    "batch_size": 512, "max_epochs": 60, "patience": 6, "early_stopping": "validation MSE on the standardized residual; restore best epoch",
    "seeds": [0, 1, 2, 3, 4],
}
LAGS = list(range(120, -1, -5))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


N_PARAMS = sum(p.numel() for p in CompactGRU(CONFIG["hidden_size"], CONFIG["num_layers"]).parameters())


In [25]:
# Admitted windows of the defined splits; the 25-value history comes from the released one-minute grid at or before t.
folds = pd.read_parquet(WINDOWS_DIR / "fold_assignments.parquet", columns=["evaluation_protocol", "subject", "origin_time", "task_id", "split", "admitted_to_split"])
folds = folds.loc[folds["admitted_to_split"]].drop(columns="admitted_to_split")
targets = pd.read_parquet(WINDOWS_DIR / "glucose_history_samples.parquet", columns=["subject", "origin_time", "task_id", "target_value", "glucose_lag_0min"])
grid_lookup = pd.read_parquet(ALIGNED_DIR / "dynamic_aligned_1min.parquet", columns=["subject", "timestamp", "libre_gl"]).set_index(["subject", "timestamp"])["libre_gl"]
windows = folds.merge(targets, on=["subject", "origin_time", "task_id"], how="left", validate="many_to_one")
history = np.column_stack([grid_lookup.reindex(pd.MultiIndex.from_arrays(
    [windows["subject"], windows["origin_time"] - pd.Timedelta(minutes=lag)])).to_numpy() for lag in LAGS])
assert np.isfinite(history).all() and windows["target_value"].notna().all()
assert np.allclose(history[:, -1], windows["glucose_lag_0min"])
windows = windows.reset_index(drop=True)
WINDOW_KEYS = ["evaluation_protocol", "subject", "origin_time", "task_id"]
# Each window belongs to exactly one split within a protocol.
assert not windows.duplicated(WINDOW_KEYS).any()

def metrics(frame: pd.DataFrame, pred: str) -> dict:
    """Subject-equal (per-subject then averaged) and pooled-window MAE / RMSE, as in the core evaluation."""

    err = frame[pred] - frame["target_value"]
    per_subject = err.groupby(frame["subject"]).agg(mae=lambda e: e.abs().mean(), rmse=lambda e: np.sqrt((e ** 2).mean()))
    return {"subject_equal_mae": per_subject["mae"].mean(), "subject_equal_rmse": per_subject["rmse"].mean(),
            "pooled_window_mae": err.abs().mean(), "pooled_window_rmse": np.sqrt((err ** 2).mean()),
            "n_windows": len(frame), "n_subjects": frame["subject"].nunique()}

In [26]:
# Training, prediction and result assembly shared by the GRU and the TCN (same windows, training rules and seeds).
def run_model(model, x_std, device):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(torch.as_tensor(x_std[i:i + 8192, :, None], dtype=torch.float32, device=device))
                          for i in range(0, len(x_std), 8192)]).cpu().numpy()


def train_sequence_models(make_model, config, device):
    """One model per protocol and seed: training split only, validation early stopping, best-epoch weights restored."""

    states, training_log, val_rows, val_predictions = {}, [], [], []
    for protocol in PROTOCOLS:
        train = windows.loc[windows["evaluation_protocol"].eq(protocol) & windows["split"].eq("train")]
        validation_windows = windows.loc[windows["evaluation_protocol"].eq(protocol) & windows["split"].eq("validation")]
        h_train, h_val = history[train.index], history[validation_windows.index]
        stats = {"history_mean": h_train.mean(), "history_sd": h_train.std(),
                 "residual_sd": (train["target_value"].to_numpy() - h_train[:, -1]).std()}
        x_train = torch.as_tensor(((h_train - stats["history_mean"]) / stats["history_sd"])[:, :, None], dtype=torch.float32, device=device)
        y_train = torch.as_tensor((train["target_value"].to_numpy() - h_train[:, -1]) / stats["residual_sd"], dtype=torch.float32, device=device)
        x_val_std = (h_val - stats["history_mean"]) / stats["history_sd"]
        y_val = (validation_windows["target_value"].to_numpy() - h_val[:, -1]) / stats["residual_sd"]
        for seed in config["seeds"]:
            torch.manual_seed(seed); np.random.seed(seed)
            model = make_model().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
            generator = torch.Generator(device="cpu").manual_seed(seed)
            best, best_epoch, best_state, wait, start = np.inf, 0, None, 0, time.perf_counter()
            for epoch in range(1, config["max_epochs"] + 1):
                model.train()
                order = torch.randperm(len(y_train), generator=generator).to(device)
                for i in range(0, len(order), config["batch_size"]):
                    idx = order[i:i + config["batch_size"]]
                    optimizer.zero_grad()
                    loss = torch.mean((model(x_train[idx]) - y_train[idx]) ** 2)
                    loss.backward()
                    optimizer.step()
                val_mse = float(np.mean((run_model(model, x_val_std, device) - y_val) ** 2))
                if val_mse < best:
                    best, best_epoch, wait = val_mse, epoch, 0
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                else:
                    wait += 1
                    if wait >= config["patience"]:
                        break
            model.load_state_dict(best_state)
            states[(protocol, seed)] = (best_state, stats)
            pred = h_val[:, -1] + run_model(model, x_val_std, device) * stats["residual_sd"]
            val_predictions.append(validation_windows[WINDOW_KEYS + ["split", "target_value"]].assign(seed=seed, prediction=pred))
            val_rows.append({"evaluation_protocol": protocol, "seed": seed, **metrics(validation_windows.assign(prediction=pred), "prediction")})
            training_log.append({"evaluation_protocol": protocol, "seed": seed, "epochs_run": epoch, "best_epoch": best_epoch,
                                 "best_validation_mse_standardized": best, "train_seconds": time.perf_counter() - start,
                                 "train_windows": len(train), "validation_windows": len(validation_windows), **{f"stat_{k}": v for k, v in stats.items()}})
    return states, pd.DataFrame(training_log), pd.DataFrame(val_rows), val_predictions


def predict_test_windows(make_model, config, states, device):
    """Each seed predicts the test windows once with its best-epoch weights."""

    test_rows, test_predictions = [], []
    for protocol in PROTOCOLS:
        test = windows.loc[windows["evaluation_protocol"].eq(protocol) & windows["split"].eq("test")]
        h_test = history[test.index]
        for seed in config["seeds"]:
            state, stats = states[(protocol, seed)]
            model = make_model().to(device)
            model.load_state_dict(state)
            pred = h_test[:, -1] + run_model(model, (h_test - stats["history_mean"]) / stats["history_sd"], device) * stats["residual_sd"]
            test_predictions.append(test[WINDOW_KEYS + ["split", "target_value"]].assign(seed=seed, prediction=pred))
            test_rows.append({"evaluation_protocol": protocol, "seed": seed, **metrics(test.assign(prediction=pred), "prediction")})
    return pd.DataFrame(test_rows), test_predictions


def subject_rmse(pred: pd.DataFrame) -> pd.Series:
    se = (pred["prediction"] - pred["target_value"]) ** 2
    return (se.groupby([pred["seed"], pred["subject"]]).mean() ** 0.5).groupby("subject").mean()


def sequence_result_records(config, n_params, device_label, training_log, val_metrics, test_metrics, comparison, paired, metric_columns,
                            aggregate_record, reference_column, delta_column, paired_record):
    """Result table of one sequence model: configuration, training, per-seed and aggregate metrics, paired comparisons, cost."""

    records = []
    add = lambda **kw: records.append(kw)
    for k, v in config.items():
        add(record_type="config", key=k, value=json.dumps(v))
    add(record_type="config", key="parameter_count", value=json.dumps(n_params))
    add(record_type="config", key="device", value=json.dumps(device_label))
    add(record_type="config", key="software", value=json.dumps({"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda}))
    for r in training_log.to_dict("records"):
        add(record_type="training", evaluation_protocol=r["evaluation_protocol"], seed=r["seed"], value=json.dumps(r, default=float))
    for split, frame in [("validation", val_metrics), ("test", test_metrics)]:
        for r in frame.to_dict("records"):
            for col in metric_columns + ["n_windows", "n_subjects"]:
                add(record_type=f"{split}_metric_per_seed", evaluation_protocol=r["evaluation_protocol"], seed=r["seed"], key=col, numeric_value=float(r[col]))
    for r in comparison.to_dict("records"):
        add(record_type=f"{r['split']}_{aggregate_record}", evaluation_protocol=r["evaluation_protocol"], key=f"{r[reference_column]}|{r['metric']}",
            numeric_value=r[delta_column], value=json.dumps({k: r[k] for k in r if k not in ("split", "evaluation_protocol")}, default=float))
    for r in paired.to_dict("records"):
        add(record_type=paired_record(r), evaluation_protocol=r["evaluation_protocol"], key="d_rmse_subject_equal",
            numeric_value=r["d_rmse_subject_equal"], value=json.dumps(r, default=float))
    add(record_type="cost", key="total_training_seconds", numeric_value=float(training_log["train_seconds"].sum()))
    return pd.DataFrame(records).assign(extension_type=config["extension_type"])

In [27]:
# Training uses the training split only; early stopping and every statistic use training or validation data only.
make_gru = lambda: CompactGRU(CONFIG["hidden_size"], CONFIG["num_layers"])
states, training_log, val_metrics, val_predictions = train_sequence_models(make_gru, CONFIG, DEVICE)

In [28]:
# The configuration and the best-epoch weights are fixed above; test windows are predicted once per seed.
test_metrics, test_predictions = predict_test_windows(make_gru, CONFIG, states, DEVICE)

In [29]:
# Baseline models on identical windows: validation metrics from `04_baseline_models.ipynb` and TEST metrics from `06_evaluation_report.ipynb` (training-split fits for all models).
BASELINES = ["glucobench_linear_regression", "persistence", "ridge_glucose_history_features"]
val_base = pd.read_csv(BASELINE_DIR / "validation_comparison.csv")
val_base = val_base.loc[val_base["scope"].eq("summary") & val_base["model_id"].isin(BASELINES)]
test_base = pd.read_csv(ART06 / "metrics_summary.csv")
test_base = test_base.loc[test_base["source"].eq("baseline registry") & test_base["model_id"].isin(BASELINES)]
reference = pd.read_parquet(ART06 / "test_predictions.parquet", columns=["experiment_id", "evaluation_protocol", "subject", "origin_time", "task_id"])
reference = reference.loc[reference["experiment_id"].str.startswith("baseline|glucobench_linear_regression")]
seq_test_keys = pd.concat(test_predictions).loc[lambda d: d["seed"].eq(0), ["evaluation_protocol", "subject", "origin_time", "task_id"]]
TEST_KEYS_MATCH = len(seq_test_keys.merge(reference, how="inner")) == len(seq_test_keys) == len(reference)
assert TEST_KEYS_MATCH
assert (val_base.groupby("evaluation_protocol")["windows"].first().sort_index()
        == val_metrics.groupby("evaluation_protocol")["n_windows"].first().sort_index()).all()

COLS = ["subject_equal_mae", "subject_equal_rmse", "pooled_window_mae", "pooled_window_rmse"]
comparison = []
for split, seq, base, cmap in [
        ("validation", val_metrics, val_base, {"subject_mean_mae": "subject_equal_mae", "subject_mean_rmse": "subject_equal_rmse",
                                              "pooled_mae": "pooled_window_mae", "pooled_rmse": "pooled_window_rmse"}),
        ("test", test_metrics, test_base, {c: c for c in COLS})]:
    base = base.rename(columns=cmap)
    for protocol in PROTOCOLS:
        s = seq.loc[seq["evaluation_protocol"].eq(protocol)]
        for model_id in BASELINES:
            b = base.loc[base["evaluation_protocol"].eq(protocol) & base["model_id"].eq(model_id)].iloc[0]
            for col in COLS:
                comparison.append({"split": split, "evaluation_protocol": protocol, "baseline": model_id, "metric": col,
                                   "gru_seed_mean": s[col].mean(), "gru_seed_sd": s[col].std(), "gru_seed_min": s[col].min(),
                                   "gru_seed_max": s[col].max(), "baseline_value": b[col], "delta_gru_minus_baseline": s[col].mean() - b[col],
                                   "seeds_better_than_baseline": int((s[col] < b[col]).sum())})
comparison = pd.DataFrame(comparison)

# Paired subject-level comparison with GlucoBench LR on test: per-subject RMSE averaged over seeds, subject cluster bootstrap
# with the evaluation settings (2000 resamples, seed 42, 95% percentile).
lock = json.loads((MANUAL_DIR / "evaluation_plan.json").read_text(encoding="utf-8"))["evaluation"]["uncertainty"]
by_subject = pd.read_csv(ART06 / "metrics_by_subject.csv")
paired_rows = []
for protocol in PROTOCOLS:
    seq = pd.concat(test_predictions).loc[lambda d: d["evaluation_protocol"].eq(protocol)]
    gru_rmse = subject_rmse(seq)
    base_rmse = by_subject.loc[by_subject["experiment_id"].eq(f"baseline|glucobench_linear_regression|{protocol}")].set_index("subject")["rmse"]
    d = (gru_rmse - base_rmse.loc[gru_rmse.index]).to_numpy()
    boot = d[draw_subjects(len(d), int(lock["resamples"]), int(lock["seed"]))].mean(axis=1)
    paired_rows.append({"evaluation_protocol": protocol, "n_subjects": len(d), "d_rmse_subject_equal": d.mean(),
                        "d_rmse_ci_low": np.percentile(boot, 2.5), "d_rmse_ci_high": np.percentile(boot, 97.5),
                        "improved_subjects": int((d < 0).sum())})
paired = pd.DataFrame(paired_rows)
with pd.option_context("display.max_columns", 20):
    display(comparison.loc[comparison["baseline"].eq("glucobench_linear_regression")].round(3))
display(paired.round(3))

,split,evaluation_protocol,baseline,metric,gru_seed_mean,gru_seed_sd,gru_seed_min,gru_seed_max,baseline_value,delta_gru_minus_baseline,seeds_better_than_baseline
0,validation,Within-subject Future Prediction,glucobench_linear_regression,subject_equal_mae,7.334,0.054,7.271,7.413,7.567,-0.234,5
1,validation,Within-subject Future Prediction,glucobench_linear_regression,subject_equal_rmse,10.476,0.051,10.418,10.537,11.075,-0.599,5
2,validation,Within-subject Future Prediction,glucobench_linear_regression,pooled_window_mae,7.192,0.072,7.076,7.272,7.481,-0.289,5
3,validation,Within-subject Future Prediction,glucobench_linear_regression,pooled_window_rmse,10.712,0.069,10.594,10.776,11.485,-0.773,5
12,validation,Unseen-subject Generalization,glucobench_linear_regression,subject_equal_mae,7.945,0.104,7.826,8.111,8.342,-0.396,5
13,validation,Unseen-subject Generalization,glucobench_linear_regression,subject_equal_rmse,11.792,0.129,11.621,11.944,12.379,-0.587,5
14,validation,Unseen-subject Generalization,glucobench_linear_regression,pooled_window_mae,7.976,0.098,7.849,8.116,8.186,-0.210,5
15,validation,Unseen-subject Generalization,glucobench_linear_regression,pooled_window_rmse,11.931,0.106,11.797,12.046,12.378,-0.447,5
24,test,Within-subject Future Prediction,glucobench_linear_regression,subject_equal_mae,7.180,0.063,7.088,7.264,7.389,-0.209,5
25,test,Within-subject Future Prediction,glucobench_linear_regression,subject_equal_rmse,10.436,0.052,10.348,10.485,10.832,-0.397,5


,evaluation_protocol,n_subjects,d_rmse_subject_equal,d_rmse_ci_low,d_rmse_ci_high,improved_subjects
0,Within-subject Future Prediction,45,-0.397,-0.614,-0.211,33
1,Unseen-subject Generalization,9,-0.488,-1.431,0.130,5


In [30]:
# One results table (configuration, training, per-seed and aggregate metrics, baselines, paired comparison, cost) plus predictions.
gru_results = sequence_result_records(
    CONFIG, N_PARAMS, DEVICE + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""), training_log, val_metrics, test_metrics,
    comparison, paired, COLS, aggregate_record="aggrefactor_checksnd_baseline", reference_column="baseline", delta_column="delta_gru_minus_baseline",
    paired_record=lambda r: "test_paired_vs_glucobench_linear_regression")
gru_predictions = pd.concat(val_predictions + test_predictions, ignore_index=True)

Per-seed metrics and paired comparisons with GlucoBench LR are shown above; all seeds are retained.


### TCN model

The causal TCN uses the same windows, targets, normalization, training budget and seeds as the GRU. Its four residual blocks have 16 channels, kernel size 3 and dilations 1, 2, 4 and 8 (31-step receptive field), with no dropout and a linear head on the last step. CPU execution provides deterministic training in the recorded environment. Both architectures are reported; these results do not select between them.


In [31]:
# Causal TCN configuration; data, training rules and seeds match the GRU.
TCN_CONFIG = {
    "extension_type": "compact_sequence_model", "model_family": "TCN", "architecture": "causal temporal convolutional network",
    "input": "25 released Libre values at t-120, t-115, ..., t (5-min steps), standardized with training-split statistics",
    "output": "g0 + residual; the residual (target - g0) is standardized with training-split statistics",
    "channels": 16, "kernel_size": 3, "dilations": [1, 2, 4, 8], "blocks": "one causal convolution + ReLU per dilation, residual connection",
    "dropout": 0.0, "head": "Linear(16, 1) on the last time step", "receptive_field_steps": 1 + 2 * (1 + 2 + 4 + 8),
    "loss": "mean squared error on the standardized residual", "optimizer": "Adam", "learning_rate": 1e-3, "weight_decay": 0.0,
    "batch_size": 512, "max_epochs": 60, "patience": 6, "early_stopping": "validation MSE on the standardized residual; restore best epoch",
    "seeds": [0, 1, 2, 3, 4],
}
assert TCN_CONFIG["receptive_field_steps"] >= 25
# TCN is trained on CPU for deterministic execution in this environment.
DEVICE_T = "cpu"
CPU_THREADS_T = torch.get_num_threads()


N_PARAMS_T = sum(p.numel() for p in CompactTCN(TCN_CONFIG["channels"], TCN_CONFIG["kernel_size"], TCN_CONFIG["dilations"]).parameters())


In [32]:
# Training split only; validation early stopping; statistics from the training split only.
make_tcn = lambda: CompactTCN(TCN_CONFIG["channels"], TCN_CONFIG["kernel_size"], TCN_CONFIG["dilations"])
tcn_states, tcn_log, tcn_val_metrics, tcn_val_pred = train_sequence_models(make_tcn, TCN_CONFIG, DEVICE_T)

In [33]:
# Configuration and best-epoch weights are fixed above; each seed predicts the test windows once.
tcn_test_metrics, tcn_test_pred = predict_test_windows(make_tcn, TCN_CONFIG, tcn_states, DEVICE_T)

In [34]:
# TCN against the GRU and the baseline models on identical windows; no model is selected from these results.
COLS_T = ["subject_equal_mae", "subject_equal_rmse", "pooled_window_mae", "pooled_window_rmse"]
BASELINES_T = ["glucobench_linear_regression", "ridge_glucose_history_features", "persistence"]
val_base_t = pd.read_csv(BASELINE_DIR / "validation_comparison.csv").rename(columns={
    "subject_mean_mae": "subject_equal_mae", "subject_mean_rmse": "subject_equal_rmse", "pooled_mae": "pooled_window_mae", "pooled_rmse": "pooled_window_rmse"})
val_base_t = val_base_t.loc[val_base_t["scope"].eq("summary") & val_base_t["model_id"].isin(BASELINES_T)]
test_base_t = pd.read_csv(ART06 / "metrics_summary.csv")
test_base_t = test_base_t.loc[test_base_t["source"].eq("baseline registry") & test_base_t["model_id"].isin(BASELINES_T)]
gru_metrics = pd.concat([val_metrics.assign(split="validation"), test_metrics.assign(split="test")], ignore_index=True)
tcn_metrics = pd.concat([tcn_val_metrics.assign(split="validation"), tcn_test_metrics.assign(split="test")], ignore_index=True)

comparison_t = []
for split in ["validation", "test"]:
    for protocol in PROTOCOLS:
        t = tcn_metrics.loc[tcn_metrics["split"].eq(split) & tcn_metrics["evaluation_protocol"].eq(protocol)]
        g = gru_metrics.loc[gru_metrics["split"].eq(split) & gru_metrics["evaluation_protocol"].eq(protocol)]
        base = (val_base_t if split == "validation" else test_base_t).loc[lambda d: d["evaluation_protocol"].eq(protocol)].set_index("model_id")
        for col in COLS_T:
            refs = {"GRU": g[col].mean(), **{m: base.at[m, col] for m in BASELINES_T}}
            for ref, value in refs.items():
                comparison_t.append({"split": split, "evaluation_protocol": protocol, "comparator": ref, "metric": col,
                                     "tcn_seed_mean": t[col].mean(), "tcn_seed_sd": t[col].std(), "tcn_seed_min": t[col].min(), "tcn_seed_max": t[col].max(),
                                     "comparator_value": value, "delta_tcn_minus_comparator": t[col].mean() - value,
                                     "seeds_better_than_comparator": int((t[col] < value).sum())})
comparison_t = pd.DataFrame(comparison_t)

# Paired subject-level comparison with the GRU and the baseline models on test: per-subject RMSE averaged over seeds, subject cluster bootstrap
# with the evaluation settings (2000 resamples, seed 42, 95% percentile).
lock_t = json.loads((MANUAL_DIR / "evaluation_plan.json").read_text(encoding="utf-8"))["evaluation"]["uncertainty"]
by_subject_t = pd.read_csv(ART06 / "metrics_by_subject.csv")


paired_t = []
tcn_test_all = pd.concat(tcn_test_pred)
for protocol in PROTOCOLS:
    tcn_rmse = subject_rmse(tcn_test_all.loc[tcn_test_all["evaluation_protocol"].eq(protocol)])
    refs = {"GRU": subject_rmse(gru_predictions.loc[gru_predictions["split"].eq("test") & gru_predictions["evaluation_protocol"].eq(protocol)])}
    refs.update({m: by_subject_t.loc[by_subject_t["experiment_id"].eq(f"baseline|{m}|{protocol}")].set_index("subject")["rmse"] for m in BASELINES_T})
    for ref, ref_rmse in refs.items():
        d = (tcn_rmse - ref_rmse.loc[tcn_rmse.index]).to_numpy()
        boot = d[draw_subjects(len(d), int(lock_t["resamples"]), int(lock_t["seed"]))].mean(axis=1)
        paired_t.append({"evaluation_protocol": protocol, "comparator": ref, "n_subjects": len(d), "d_rmse_subject_equal": d.mean(),
                         "d_rmse_ci_low": np.percentile(boot, 2.5), "d_rmse_ci_high": np.percentile(boot, 97.5), "improved_subjects": int((d < 0).sum())})
paired_t = pd.DataFrame(paired_t)
display(comparison_t.loc[comparison_t["metric"].isin(["subject_equal_mae", "subject_equal_rmse"])].round(3))
display(paired_t.round(3))

,split,evaluation_protocol,comparator,metric,tcn_seed_mean,tcn_seed_sd,tcn_seed_min,tcn_seed_max,comparator_value,delta_tcn_minus_comparator,seeds_better_than_comparator
0,validation,Within-subject Future Prediction,GRU,subject_equal_mae,7.347,0.115,7.225,7.487,7.334,0.013,2
1,validation,Within-subject Future Prediction,glucobench_linear_regression,subject_equal_mae,7.347,0.115,7.225,7.487,7.567,-0.220,5
2,validation,Within-subject Future Prediction,ridge_glucose_history_features,subject_equal_mae,7.347,0.115,7.225,7.487,7.974,-0.627,5
3,validation,Within-subject Future Prediction,persistence,subject_equal_mae,7.347,0.115,7.225,7.487,10.616,-3.269,5
4,validation,Within-subject Future Prediction,GRU,subject_equal_rmse,10.511,0.076,10.405,10.582,10.476,0.034,2
5,validation,Within-subject Future Prediction,glucobench_linear_regression,subject_equal_rmse,10.511,0.076,10.405,10.582,11.075,-0.564,5
6,validation,Within-subject Future Prediction,ridge_glucose_history_features,subject_equal_rmse,10.511,0.076,10.405,10.582,11.477,-0.967,5
7,validation,Within-subject Future Prediction,persistence,subject_equal_rmse,10.511,0.076,10.405,10.582,15.443,-4.932,5
16,validation,Unseen-subject Generalization,GRU,subject_equal_mae,8.000,0.062,7.936,8.088,7.945,0.055,1
17,validation,Unseen-subject Generalization,glucobench_linear_regression,subject_equal_mae,8.000,0.062,7.936,8.088,8.342,-0.342,5


,evaluation_protocol,comparator,n_subjects,d_rmse_subject_equal,d_rmse_ci_low,d_rmse_ci_high,improved_subjects
0,Within-subject Future Prediction,GRU,45,0.017,-0.045,0.081,21
1,Within-subject Future Prediction,glucobench_linear_regression,45,-0.380,-0.618,-0.170,31
2,Within-subject Future Prediction,ridge_glucose_history_features,45,-0.734,-0.959,-0.531,39
3,Within-subject Future Prediction,persistence,45,-4.565,-5.341,-3.848,44
4,Unseen-subject Generalization,GRU,9,-0.018,-0.077,0.042,5
5,Unseen-subject Generalization,glucobench_linear_regression,9,-0.506,-1.423,0.128,5
6,Unseen-subject Generalization,ridge_glucose_history_features,9,-0.659,-1.161,-0.247,7
7,Unseen-subject Generalization,persistence,9,-4.550,-6.165,-2.906,8


In [35]:
# Merge the TCN records into the sequence-model artifacts; GRU records are kept unchanged and labeled with their model family.
tcn_results = sequence_result_records(
    TCN_CONFIG, N_PARAMS_T, f"{DEVICE_T} ({CPU_THREADS_T} threads)", tcn_log, tcn_val_metrics, tcn_test_metrics, comparison_t, paired_t, COLS_T,
    aggregate_record="aggrefactor_checksnd_comparator", reference_column="comparator", delta_column="delta_tcn_minus_comparator",
    paired_record=lambda r: f"test_paired_vs_{r['comparator']}")

FAMILY = {"GRU": "recurrent sequence model", "TCN": "convolutional temporal model"}
merged_results = pd.concat([gru_results.assign(model_family="GRU"), tcn_results.assign(model_family="TCN")], ignore_index=True)
merged_results["architecture_family"] = merged_results["model_family"].map(FAMILY)
tcn_predictions = pd.concat(tcn_val_pred + tcn_test_pred, ignore_index=True)
merged_predictions = pd.concat([gru_predictions.assign(model_family="GRU"), tcn_predictions.assign(model_family="TCN")], ignore_index=True)
merged_results.to_csv(EXT_DIR / "sequence_model_results.csv", index=False, encoding="utf-8")
merged_predictions.to_parquet(EXT_DIR / "sequence_predictions.parquet", index=False)

The tables report configuration, per-seed metrics and paired comparisons with the baselines and GRU. All seeds and both architecture families are retained; TEST results are exploratory.
